# Variational Quantum Circuit

PCA features
    │
    ▼
Angle Encoding
    │
    ▼
VQC
    │
    ├── Rot gates
    ├── CNOT entanglement
    └── N layers
    │
    ▼
Measure ALL qubits
    │
    ▼
n_qubits expectation values
    │
    ▼
Classical trainable layer
    │
    ▼
4 logits
    │
    ▼
Softmax
    │
    ▼
4 classes

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor
from pennylane import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

import pennylane as qml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Experiment 2: New Architecture

In [5]:
# ============================================================
# PATHS
# ============================================================

ANGLE_ROOT = Path(
    "../data/vqc/angle_encoding"
)

RESULTS_ROOT = Path(
    "../results/vqc/experiment2"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXCEL_PATH = (
    RESULTS_ROOT /
    "vqc_pca_qubit_comparison_experiment2.xlsx"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

QUBIT_LIST = [
    4,
    8,
    12,
    16
]

N_LAYERS = 2
N_CLASSES = 4
N_EPOCHS = 30
LEARNING_RATE = 0.05
BATCH_SIZE = 32
RANDOM_SEED = 42

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

# RANDOM SEED
np.random.seed(
    RANDOM_SEED
)

# ============================================================
# FUNCTIONS
# ============================================================

def softmax(x):

    x = np.asarray(x)

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        np.sum(
            exp_x,
            axis=-1,
            keepdims=True
        )
    )

def initialise_weights(n_qubits):

    return (
        0.01
        * np.random.randn(
            N_LAYERS,
            n_qubits,
            3
        )
    )

def initialise_classical_layer(n_qubits):

    W = (
        0.01
        * np.random.randn(
            n_qubits,
            N_CLASSES
        )
    )

    b = np.zeros(
        N_CLASSES
    )

    return W, b

def create_quantum_circuit(n_qubits):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev)
    def quantum_circuit(
        x,
        weights
    ):

        # ANGLE ENCODING
        qml.AngleEmbedding(
            x,
            wires=range(n_qubits),
            rotation="Y"
        )

        for layer in range(N_LAYERS):

            # TRAINABLE ROTATIONS
            for qubit in range(n_qubits):

                # Each Rot gate has three trainable parameters.
                # So 2 layers × 8 qubits × 3 = 48 trainable parameters.
                qml.Rot(
                    weights[layer, qubit, 0],
                    weights[layer, qubit, 1],
                    weights[layer, qubit, 2],
                    wires=qubit
                )

            # ENTANGLEMENT
            for qubit in range(
                n_qubits - 1
            ):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1
                    ]
                )

        # MEASUREMENTS
        return [
            qml.expval(
                qml.PauliZ(qubit)
            )
            for qubit in range(n_qubits)
        ]

    return quantum_circuit

# FORWARD PASS

def predict_logits(
    X,
    quantum_weights,
    classical_weights,
    classical_bias,
    quantum_circuit
):

    outputs = []

    for sample in X:

        quantum_output = quantum_circuit(
            sample,
            quantum_weights
        )

        quantum_output = np.asarray(
            quantum_output
        )

        logits = (
            quantum_output
            @ classical_weights
            + classical_bias
        )

        outputs.append(
            logits
        )

    return np.asarray(
        outputs
    )


# CROSS-ENTROPY

def cross_entropy(
    probabilities,
    labels
):

    probabilities = np.clip(
        probabilities,
        1e-10,
        1.0
    )

    losses = -np.log(
        probabilities[
            np.arange(
                len(labels)
            ),
            labels
        ]
    )

    return np.mean(
        losses
    )

def prediction_distribution(predictions):

    counts = np.bincount(
        predictions,
        minlength=N_CLASSES
    )

    percentages = (
        counts
        /
        len(predictions)
        *
        100
    )

    return counts, percentages

# ============================================================
# TRAIN VQC
# ============================================================

def train_vqc(
    X_train,
    y_train,
    n_qubits
):

    quantum_circuit = (
        create_quantum_circuit(
            n_qubits
        )
    )

    quantum_weights = (
        initialise_weights(
            n_qubits
        )
    )

    classical_weights, classical_bias = (
        initialise_classical_layer(
            n_qubits
        )
    )

    # OPTIMISER
    opt = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )

    # COST FUNCTION
    def cost_fn(
        quantum_weights,
        classical_weights,
        classical_bias,
        X_batch,
        y_batch
    ):

        logits = predict_logits(
            X_batch,
            quantum_weights,
            classical_weights,
            classical_bias,
            quantum_circuit
        )

        probabilities = softmax(
            logits
        )

        return cross_entropy(
            probabilities,
            y_batch
        )

    # TRAINING LOOP

    loss_history = []

    for epoch in range(
        N_EPOCHS
    ):

        # Random mini-batch
        batch_size = min(
            BATCH_SIZE,
            len(X_train)
        )

        batch_indices = np.random.choice(
            len(X_train),
            size=batch_size,
            replace=False
        )

        X_batch = X_train[
            batch_indices
        ]

        y_batch = y_train[
            batch_indices
        ]

        (
            quantum_weights,
            classical_weights,
            classical_bias
        ), loss = opt.step_and_cost(

            lambda qw, cw, cb:
                cost_fn(
                    qw,
                    cw,
                    cb,
                    X_batch,
                    y_batch
                ),

            quantum_weights,
            classical_weights,
            classical_bias
        )

        loss_history.append(
            float(loss)
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:3d}/{N_EPOCHS} "
                f"| Loss = "
                f"{loss:.6f}"
            )

    return (
        quantum_weights,
        classical_weights,
        classical_bias,
        quantum_circuit,
        loss_history
    )

# ============================================================
# PREDICTION
# ============================================================

def predict(
    X,
    quantum_weights,
    classical_weights,
    classical_bias,
    quantum_circuit
):

    logits = predict_logits(
        X,
        quantum_weights,
        classical_weights,
        classical_bias,
        quantum_circuit
    )

    probabilities = softmax(
        logits
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return (
        predictions,
        probabilities
    )

all_results = []

for n_qubits in QUBIT_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC EXPERIMENT: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)

    # Check angle encoding directory

    for test_subject in SUBJECTS:

        print("\n")
        print("-" * 80)

        print(
            f"QUBITS = {n_qubits}"
        )

        print(
            f"TEST SUBJECT = "
            f"{test_subject}"
        )

        print("-" * 80)

        # Load corresponding angle data

        angle_path = (
            ANGLE_ROOT
            /
            f"loso_test_{test_subject}"
            /
            f"angle_{n_qubits}"
            /
            "data.npz"
        )

        if not angle_path.exists():

            raise FileNotFoundError(
                f"\nMissing angle file:\n"
                f"{angle_path}\n\n"
                f"Make sure PCA/angle encoding "
                f"has been generated for "
                f"{n_qubits} components."
            )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        X_train = data[
            "angles_train"
        ]

        X_test = data[
            "angles_test"
        ]

        y_train = data[
            "y_train"
        ]

        y_test = data[
            "y_test"
        ]

        # Check dimensions

        if X_train.shape[1] != n_qubits:

            raise ValueError(
                f"Expected {n_qubits} "
                f"features but received "
                f"{X_train.shape[1]}"
            )

        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )

        print(
            "Training labels:",
            np.unique(y_train)
        )

        print(
            "Testing labels:",
            np.unique(y_test)
        )

        # TRAIN

        print("\n")
        print(
            "Training VQC..."
        )

        (
            quantum_weights,
            classical_weights,
            classical_bias,
            quantum_circuit,
            loss_history
        ) = train_vqc(
            X_train,
            y_train,
            n_qubits
        )

        # TEST

        print("\n")
        print(
            "Testing VQC..."
        )

        predictions, probabilities = (
            predict(
                X_test,
                quantum_weights,
                classical_weights,
                classical_bias,
                quantum_circuit
            )
        )

        prediction_counts, prediction_percentages = (
            prediction_distribution(
                predictions
            )
        )

        prediction_collapse = np.max(
            prediction_percentages
        )

        # METRICS

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )

        macro_f1 = f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=[
                0,
                1,
                2,
                3
            ]
        )

        print("\nPrediction distribution:")

        for class_id in range(N_CLASSES):

            print(
                f"Class {class_id}: "
                f"{prediction_counts[class_id]} "
                f"("
                f"{prediction_percentages[class_id]:.2f}%"
                f")"
            )

        print(
            "\nTest accuracy:",
            f"{accuracy:.4f}"
        )

        print(
            "Balanced accuracy:",
            f"{balanced_accuracy:.4f}"
        )

        print(
            "Macro F1:",
            f"{macro_f1:.4f}"
        )

        print(
            "Weighted F1:",
            f"{weighted_f1:.4f}"
        )

        print(
            "\nConfusion matrix:"
        )

        print(cm)

        # SAVE INDIVIDUAL RESULT

        result_dir = (
            RESULTS_ROOT
            /
            f"qubits_{n_qubits}"
        )

        result_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            result_dir
            /
            f"loso_test_{test_subject}.npz"
        )

        np.savez_compressed(

            save_path,
            test_subject=test_subject,
            predictions=predictions,
            prediction_collapse=prediction_collapse,
            probabilities=probabilities,
            y_test=y_test,
            accuracy=accuracy,
            balanced_accuracy=balanced_accuracy,
            macro_f1=macro_f1,
            weighted_f1=weighted_f1,
            confusion_matrix=cm,
            quantum_weights=quantum_weights,
            classical_weights=classical_weights,
            classical_bias=classical_bias,
            loss_history=np.asarray(
                loss_history
            ),
            n_qubits=n_qubits,
            n_layers=N_LAYERS
        )

        print(
            "Saved:",
            save_path
        )

        # STORE ROW FOR EXCEL

        all_results.append({

            "subject": test_subject,
            "n_qubits": n_qubits,
            "accuracy": accuracy,
            "balanced_accuracy":
                balanced_accuracy,
            "macro_f1":
                macro_f1,
            "weighted_f1":
                weighted_f1,
            "final_training_loss":
                loss_history[-1],
            "prediction_collapse":
                prediction_collapse
        })

# CONVERT RESULTS TO DATAFRAME

results_df = pd.DataFrame(
    all_results
)
    
# LOSO SUMMARY

summary_df = (
    results_df
    .groupby(
        "n_qubits"
    )
    .agg({

        "accuracy":
            ["mean", "std"],

        "balanced_accuracy":
            ["mean", "std"],

        "macro_f1":
            ["mean", "std"],

        "weighted_f1":
            ["mean", "std"],

        "final_training_loss":
            ["mean", "std"]

    })
    .reset_index()
)

# Flatten column names

summary_df.columns = [

    "n_qubits",

    "accuracy_mean",
    "accuracy_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "macro_f1_mean",
    "macro_f1_std",

    "weighted_f1_mean",
    "weighted_f1_std",

    "loss_mean",
    "loss_std"

]

# SAVE EXCEL

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="LOSO Results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


print("\n")
print("=" * 80)
print("EXCEL RESULTS SAVED")
print("=" * 80)

print(
    EXCEL_PATH
)

# PRINT SUMMARY

print("\n")
print("=" * 80)
print("VQC QUANTUM DIMENSION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)

# ============================================================
# PLOT 1:
# MEAN ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["accuracy_mean"],
    yerr=summary_df["accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "VQC Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


accuracy_plot = (
    RESULTS_ROOT /
    "accuracy_vs_qubits.png"
)

plt.savefig(
    accuracy_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 2:
# MACRO F1 VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["macro_f1_mean"],
    yerr=summary_df["macro_f1_std"],
    marker="o",
    capsize=5
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "VQC Macro F1 vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()


f1_plot = (
    RESULTS_ROOT /
    "macro_f1_vs_qubits.png"
)

plt.savefig(
    f1_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 3:
# BALANCED ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["balanced_accuracy_mean"],
    yerr=summary_df["balanced_accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Balanced accuracy"
)

plt.title(
    "VQC Balanced Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


balanced_plot = (
    RESULTS_ROOT /
    "balanced_accuracy_vs_qubits.png"
)

plt.savefig(
    balanced_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 4:
# ACCURACY FOR EACH SUBJECT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = results_df[
        results_df["subject"] == subject
    ]

    plt.plot(
        subject_data["n_qubits"],
        subject_data["accuracy"],
        marker="o",
        label=f"Subject {subject}"
    )


plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "LOSO Accuracy Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


subject_plot = (
    RESULTS_ROOT /
    "accuracy_by_subject.png"
)

plt.savefig(
    subject_plot,
    dpi=300
)

plt.close()


# ============================================================
# COMPLETE
# ============================================================

print("\n")
print("=" * 80)
print("ALL VQC EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Plots saved in:",
    RESULTS_ROOT
)



VQC EXPERIMENT: 4 QUBITS


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 002
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.387654
Epoch   5/30 | Loss = 1.412761
Epoch  10/30 | Loss = 1.431214
Epoch  15/30 | Loss = 1.403543
Epoch  20/30 | Loss = 1.358173
Epoch  25/30 | Loss = 1.344019
Epoch  30/30 | Loss = 1.399330


Testing VQC...

Prediction distribution:
Class 0: 0 (0.00%)
Class 1: 1484 (100.00%)
Class 2: 0 (0.00%)
Class 3: 0 (0.00%)

Test accuracy: 0.2729
Balanced accuracy: 0.2500
Macro F1: 0.1715
Weighted F1: 0.1839

Confusion matrix:
[[  0 400   0   0]
 [  0 405   0   0]
 [  0 217   0   0]
 [  0 462   0   0]]
Saved: ../results/vqc/experiment2/qubits_4/loso_test_002.npz


--------------------------------------------------------------------------------
QU